# QRT Asset Allocation - equal3 ensemble

Predict whether each allocation's next return is positive (`prediction = 1[target > 0]`).
The metric is accuracy. The model averages the probabilities of three members (logistic
regression, ridge on `asinh(target)`, LightGBM), all fitted on the same features, with a 0.5 threshold.

Rules applied throughout:
- a `TS` (date) is never split between train and validation, because the allocations of a date share a common factor;
- `TS` is only used for grouping: it is not a feature, and no chronology is assumed;
- `ROW_ID` is only used to align the submission (a 1-NN on `ROW_ID` scores 100% in-sample: pure memorization);
- every statistic (imputation, scaling, one-hot categories) is learned on the training fold only.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from scipy import sparse
from scipy.stats import t as student_t
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", message="X does not have valid feature names")
DATA = Path("data")  # the challenge files X_train, y_train and X_test, next to this notebook

X = pd.read_csv(DATA / "X_train_9xQjqvZ.csv")
train = X.merge(pd.read_csv(DATA / "y_train_Ppwhaz8.csv"), on="ROW_ID", validate="one_to_one")
train["label"] = (train["target"] > 0).astype(np.int8)
test = pd.read_csv(DATA / "X_test_1zTtEnD.csv")

assert not set(train["TS"]) & set(test["TS"]), "no TS shared between train and test"
assert not train.duplicated(["TS", "ALLOCATION"]).any()
print(f"train {train.shape}, {train['TS'].nunique()} TS | test {test.shape}, {test['TS'].nunique()} TS | "
      f"positive rate {train['label'].mean():.4f}")

train (527073, 47), 2522 TS | test (31870, 45), 120 TS | positive rate 0.5072


## Fold-local features

Raw variables are median-imputed (plus missing-value indicators) and standardized, with one-hot
`GROUP`/`ALLOCATION`. On top of that come 18 **cross-sectional** features. For 5 sources (`RET_1`,
mean of the last 3 returns, volatility, share of missing volumes, `log1p(turnover)`), they give the rank,
the distance to the median and the MAD z-score **within the current date**, plus 3 interactions.
They assume that the whole date is observed before predicting.

In [2]:
RET = [f"RET_{i}" for i in range(1, 21)]
VOL = [f"SIGNED_VOLUME_{i}" for i in range(1, 21)]
NUM = RET + VOL + ["MEDIAN_DAILY_TURNOVER"]


def sources(x, ret_fill, turn_fill):
    r = x[RET].to_numpy(float)
    r = np.where(np.isfinite(r), r, ret_fill)
    turn = x["MEDIAN_DAILY_TURNOVER"].to_numpy(float)
    turn = np.where(np.isfinite(turn), turn, turn_fill)
    return {"ret1": r[:, 0], "short": r[:, :3].mean(1), "vol": r.std(1),
            "miss": np.mean(~np.isfinite(x[VOL].to_numpy(float)), axis=1), "turn": np.log1p(np.maximum(turn, 0))}


def cross_section(x, fit):
    ts = x["TS"].to_numpy()
    cols, z, rank = [], {}, {}
    for name, v in sources(x, fit["ret_fill"], fit["turn_fill"]).items():
        g = pd.Series(v).groupby(ts)
        n = g.transform("size").to_numpy(float)
        rank[name] = np.divide(g.rank().to_numpy() - 1, n - 1, out=np.full(len(v), 0.5), where=n > 1) - 0.5
        delta = v - g.transform("median").to_numpy()
        mad = pd.Series(np.abs(delta)).groupby(ts).transform("median").to_numpy()
        z[name] = np.clip(delta / np.maximum(1.4826 * mad, fit["floor"][name]), -8, 8)
        cols += [rank[name], delta, z[name]]
    miss = sources(x, fit["ret_fill"], fit["turn_fill"])["miss"]
    cols += [z["ret1"] * miss, z["short"] * miss, rank["short"] * miss]
    return np.nan_to_num(np.column_stack(cols), nan=0.0, posinf=8.0, neginf=-8.0)


class Design:
    # Everything is learned in fit (training fold), then only applied in transform.
    def fit(self, x):
        ret_fill = np.nanmedian(x[RET].to_numpy(float), axis=0)
        turn_fill = float(np.nanmedian(x["MEDIAN_DAILY_TURNOVER"].to_numpy(float)))
        self.cs = {"ret_fill": np.where(np.isfinite(ret_fill), ret_fill, 0.0),
                   "turn_fill": turn_fill if np.isfinite(turn_fill) else 0.0}
        self.cs["floor"] = {k: max(1.4826 * np.median(np.abs(v - np.median(v))) * 1e-3, 1e-12)
                            for k, v in sources(x, self.cs["ret_fill"], self.cs["turn_fill"]).items()}
        self.imp = SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True).fit(x[NUM])
        self.num_scaler = StandardScaler().fit(self.imp.transform(x[NUM]))
        self.cs_scaler = StandardScaler().fit(cross_section(x, self.cs))
        self.ohe = OneHotEncoder(handle_unknown="ignore").fit(x[["GROUP", "ALLOCATION"]])
        return self

    def transform(self, x):
        num = np.hstack([self.num_scaler.transform(self.imp.transform(x[NUM])),
                         self.cs_scaler.transform(cross_section(x, self.cs))])
        cat = self.ohe.transform(x[["GROUP", "ALLOCATION"]])
        # Multiplying by the identity keeps the original sparse layout (the lsqr solver is sensitive to it).
        cat = (cat @ sparse.diags(np.ones(cat.shape[1]), format="csr")).tocsr()
        return sparse.hstack([sparse.csr_matrix(num), cat], format="csr")

## The three members

- **ridge logistic regression** (`C = 1`) on the sign;
- **`asinh` ridge**: regress `z = asinh(target / s)` with `s = median|target|`. `asinh` compresses the tails
  but keeps the sign, hence the decision boundary. Then `P(target > 0) = F_t5(z_hat / sigma)`;
- **LightGBM** with frozen hyperparameters (never tuned), weakly correlated with the two linear models.

In [3]:
LOGIT = dict(C=1.0, solver="newton-cholesky", max_iter=100, tol=1e-7, random_state=0)
LGBM = dict(objective="binary", n_estimators=250, learning_rate=0.04, num_leaves=15, min_child_samples=500,
            reg_alpha=0.25, reg_lambda=5.0, subsample=1.0, colsample_bytree=1.0, random_state=0,
            n_jobs=4, deterministic=True, force_col_wise=True, verbosity=-1)
MEMBERS = ("logit", "asinh", "lgbm")


def fit_predict(tr, ev):
    design = Design().fit(tr)
    A, B = design.transform(tr), design.transform(ev)
    y, target = tr["label"].to_numpy(np.int8), tr["target"].to_numpy(float)
    p = {"logit": LogisticRegression(**LOGIT).fit(A, y).predict_proba(B)[:, 1]}
    z = np.arcsinh(target / max(np.median(np.abs(target)), 1e-6))
    ridge = Ridge(alpha=1.0, solver="lsqr", tol=1e-6, max_iter=2000).fit(A, z)
    sigma = max(np.sqrt(np.mean((z - ridge.predict(A)) ** 2)) * np.sqrt(3 / 5), 0.05)
    p["asinh"] = np.clip(student_t.cdf(ridge.predict(B) / sigma, df=5.0), 1e-7, 1 - 1e-7)
    p["lgbm"] = LGBMClassifier(**LGBM).fit(A, y).predict_proba(B)[:, 1]
    return p


def bagged(tr, ev, seeds=(2711, 2729, 2741)):
    # Each member is fitted on 3 random subsets of 80% of the dates; scores are averaged.
    ts = np.array(sorted(tr["TS"].astype(str).unique()))
    runs = []
    for seed in seeds:
        keep = np.random.default_rng(seed).choice(ts, size=int(round(0.8 * len(ts))), replace=False)
        runs.append(fit_predict(tr[tr["TS"].isin(keep)], ev))
    return {k: np.mean([r[k] for r in runs], axis=0) for k in MEMBERS}

## Validation: 5 folds grouped by date

One repetition, without bagging (about 3 minutes). The uncertainty of the delta comes from a bootstrap
that resamples whole **dates**, since rows of the same date are not independent.

In [4]:
folds = np.array_split(np.random.default_rng(2711).permutation(np.array(sorted(train["TS"].unique()))), 5)
oof = {k: np.zeros(len(train)) for k in MEMBERS}
for val_ts in folds:
    va = train["TS"].isin(val_ts).to_numpy()
    assert not set(train.loc[~va, "TS"]) & set(val_ts)
    for k, v in fit_predict(train[~va], train[va]).items():
        oof[k][va] = v
oof["equal3"] = np.mean([oof[k] for k in MEMBERS], axis=0)


def ts_bootstrap_ci(correct, reference, ts, n=2000, seed=48637):
    codes, _ = pd.factorize(ts)
    rows = np.bincount(codes)
    diff = np.bincount(codes, weights=correct.astype(float) - reference)
    w = np.random.default_rng(seed).multinomial(len(rows), np.full(len(rows), 1 / len(rows)), size=n)
    return np.quantile((w @ diff) / (w @ rows), [0.025, 0.975])


y = train["label"].to_numpy()
correct = {k: (v >= 0.5) == y for k, v in oof.items()}
table = pd.DataFrame({k: {"accuracy": c.mean(), "delta vs logit": c.mean() - correct["logit"].mean()}
                      for k, c in correct.items()}).T
table[["CI low", "CI high"]] = [ts_bootstrap_ci(c, correct["logit"], train["TS"]) for c in correct.values()]
table.round(5)

,accuracy,delta vs logit,CI low,CI high
logit,0.52490,0.00000,0.00000,0.00000
asinh,0.52488,-0.00002,-0.00103,0.00104
lgbm,0.52531,0.00041,-0.00146,0.00231
equal3,0.52651,0.00161,0.00064,0.00258


On the **locked holdout** of 2026-09-05 (504 unseen dates, protocol fixed before the test),
equal3 scored 0.5244, against 0.5232 for the logistic regression, 0.5225 for `asinh` and 0.5234 for LightGBM.
It beats all three members, and the paired t-test over dates was significant (p = 0.006). However, the
bootstrap CI of the gain ([-0.0010, +0.0034]) includes zero, and the gain came mostly from small dates.
equal3 is therefore a **submission candidate with mixed evidence**, not a proven better model.

## Submission

In [5]:
scores = bagged(train, test)
final = np.mean([scores[k] for k in MEMBERS], axis=0)
submission = pd.DataFrame({"ROW_ID": test["ROW_ID"], "prediction": (final >= 0.5).astype(int)})

assert list(submission.columns) == ["ROW_ID", "prediction"] and len(submission) == len(test)
assert submission["ROW_ID"].is_unique and submission["prediction"].isin([0, 1]).all()
submission.to_csv("submission_equal3.csv", index=False)
print(f"{len(submission)} rows, positive prediction rate {submission['prediction'].mean():.6f}")

31870 rows, positive prediction rate 0.574898


## Limitations

- Differences between models (~0.1-0.2 pt) are much smaller than the leaderboard noise (~±1.4 pt).
- The three members were chosen after looking at results; the holdout is now used up.
- The cross-sectional features require the whole date to be available at prediction time.